In [1]:
"""
Extreme Learning Machine (ELM) on MNIST
========================================
This example:
    1. Loads MNIST using scikit-learn's fetch_openml.
    2. Builds a random hidden layer.
    3. Solves for the output weights using the NORMAL EQUATION
           beta = (H^T H)^-1 H^T T
       instead of the Moore-Penrose pseudo-inverse
       (np.linalg.pinv), which is the approach most ELM tutorials use.
"""

import time

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [2]:
class ExtremeLearningMachine:
    """
    A minimal Extreme Learning Machine for multi-class classification.

    Parameters
    ----------
    n_hidden : int
        Number of neurons in the random (untrained) hidden layer. More
        hidden neurons give a richer random feature space and typically
        higher accuracy, at the cost of a larger (n_hidden x n_hidden)
        linear system to solve.
    reg_lambda : float
        Ridge-regression penalty added to the normal equation. H^T H is
        built from random projections and can be ill-conditioned or
        even singular, so a small amount of regularization keeps the
        linear solve numerically stable. Set this to 0.0 to recover the
        textbook-pure normal equation with no regularization.
    activation : callable, optional
        Nonlinear activation applied to the hidden layer. Defaults to
        the logistic sigmoid.
    random_state : int
        Seed for the random input -> hidden weights, for reproducibility.
    """

    def __init__(self, n_hidden=1000, reg_lambda=1.0, activation=None, random_state=42):
        self.n_hidden = n_hidden
        self.reg_lambda = reg_lambda
        self.activation = activation if activation is not None else self._sigmoid
        self.rng = np.random.RandomState(random_state)

        # Populated by fit(): W and b are random and fixed; beta is the
        # only thing that is actually "learned".
        self.W = None
        self.b = None
        self.beta = None

    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-z))

    def _hidden_layer_output(self, X):
        """
        H = activation(X @ W + b)

        Each row of H is the "random feature" representation of one
        input sample: the sample is projected through a fixed random
        linear map and passed through a nonlinearity.
        """
        return self.activation(X @ self.W + self.b)

    def fit(self, X, T):
        """
        Trains the ELM.

        X : ndarray, shape (n_samples, n_features)
        T : ndarray, shape (n_samples, n_classes) -- one-hot encoded targets
        """
        n_features = X.shape[1]

        # Step 1: randomly initialize the input -> hidden weights/biases.
        # These stay fixed for the lifetime of the model.
        self.W = self.rng.uniform(-1, 1, size=(n_features, self.n_hidden))
        self.b = self.rng.uniform(-1, 1, size=(self.n_hidden,))

        # Step 2: push the training data through the random hidden layer.
        H = self._hidden_layer_output(X)

        # Step 3: solve for beta with the NORMAL EQUATION, i.e. the
        # closed-form solution of the least-squares problem
        #
        #     minimize_beta  || H @ beta - T ||^2
        #
        # which is
        #
        #     beta = (H^T H)^{-1} H^T T
        #
        # We add a ridge term (reg_lambda * I) for numerical stability
        # -- this is standard practice for ELMs ("regularized ELM"),
        # since H^T H, built from random features, can be nearly
        # singular. With reg_lambda = 0 this is exactly the plain
        # normal equation.
        #
        # We call np.linalg.solve(A, b) rather than forming the inverse
        # explicitly (np.linalg.inv(A) @ b): both are the normal-equation
        # approach (as opposed to np.linalg.pinv(H) @ T, which factors H
        # itself via SVD), but solve() avoids explicitly computing the
        # matrix inverse and is the numerically preferred way to solve
        # a linear system such as this one.
        HtH = H.T @ H
        HtT = H.T @ T
        I = np.eye(self.n_hidden)
        self.beta = np.linalg.solve(HtH + self.reg_lambda * I, HtT)

        return self

    def predict(self, X):
        """Returns predicted integer class labels for input X."""
        H = self._hidden_layer_output(X)
        scores = H @ self.beta  # shape (n_samples, n_classes)
        return np.argmax(scores, axis=1)

In [3]:
def main():
    # -----------------------------------------------------------------
    # 1. Load MNIST via scikit-learn
    # -----------------------------------------------------------------
    # fetch_openml downloads (and locally caches, so subsequent runs are
    # fast) the classic MNIST dataset: 70,000 handwritten-digit images,
    # 28x28 pixels each, already flattened into 784-length feature
    # vectors.
    print("Loading MNIST (first run downloads ~55 MB and may take a while)...")
    X, y = fetch_openml("mnist_784", version=1, return_X_y=True, as_frame=False)
    y = y.astype(int)

    # -----------------------------------------------------------------
    # 2. Preprocess
    # -----------------------------------------------------------------
    # Scale pixel values from [0, 255] to [0, 1] so that X @ W + b stays
    # in a reasonable range for the sigmoid activation.
    X = X / 255.0

    # Standard MNIST split: 60,000 training images, 10,000 test images.
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=10000, random_state=42, stratify=y
    )

    # One-hot encode the labels, e.g. digit 3 -> [0,0,0,1,0,0,0,0,0,0].
    # The ELM's output layer is trained as a linear regression onto
    # these one-hot target vectors; the predicted class is whichever
    # output has the highest score.
    encoder = OneHotEncoder(sparse_output=False)
    T_train = encoder.fit_transform(y_train.reshape(-1, 1))

    # -----------------------------------------------------------------
    # 3. Train the ELM
    # -----------------------------------------------------------------
    elm = ExtremeLearningMachine(n_hidden=1000, reg_lambda=1.0, random_state=42)

    start = time.time()
    elm.fit(X_train, T_train)
    elapsed = time.time() - start
    print(f"Training finished in {elapsed:.2f} seconds "
          f"(a single linear solve -- no epochs, no backprop).")

    # -----------------------------------------------------------------
    # 4. Evaluate
    # -----------------------------------------------------------------
    y_pred = elm.predict(X_test)
    accuracy = np.mean(y_pred == y_test)
    print(f"Test accuracy: {accuracy:.4f}")

In [5]:
if __name__ == "__main__":
    main()

Loading MNIST (first run downloads ~55 MB and may take a while)...
Training finished in 2.54 seconds (a single linear solve -- no epochs, no backprop).
Test accuracy: 0.9334
